In [2]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Battery-health-predictor/data")
df = pd.read_csv('cleaned_battery_data.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Feature Engineering**

In [3]:
df["voltage_drop"] = df["voltage_max"] - df["voltage_mean"]
df["thermal_stress"] = df["temp_max"] * df["current_mean"]
df["discharge_intensity"] = df["current_mean"] / (df["discharge_time"] + 1e-6)

df["voltage_mean_sq"] = df["voltage_mean"] ** 2
df["temp_max_sq"] = df["temp_max"] ** 2

df["inv_voltage"] = 1 / (df["voltage_mean"] + 1e-6)
df["inv_time"] = 1 / (df["discharge_time"] + 1e-6)

df["low_voltage_flag"] = (
    df["voltage_mean"] < df["voltage_mean"].quantile(0.3)
).astype(int)


**Split Features & Target**

In [4]:
x = df.drop("battery_health", axis=1)
y = df["battery_health"]
groups = df["battery_id"]
x = x.drop("battery_id", axis=1)

In [5]:
print("X columns:", x.columns.tolist())
print("Number of features:", x.shape[1])


X columns: ['voltage_mean', 'voltage_max', 'voltage_std', 'current_mean', 'current_std', 'temp_max', 'temp_mean', 'discharge_time', 'voltage_drop', 'thermal_stress', 'discharge_intensity', 'voltage_mean_sq', 'temp_max_sq', 'inv_voltage', 'inv_time', 'low_voltage_flag']
Number of features: 16


**Feature Scaling**

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x, y, groups))
x_train = x.iloc[train_idx]
x_test = x.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
print("Train features:", x_train.shape[1])
print("Test features :", x_test.shape[1])


Train features: 16
Test features : 16


**Save Train/Test Data**

In [8]:
np.save("../data/processed/X_train.npy", x_train)
np.save("../data/processed/X_test.npy", x_test)
np.save("../data/processed/y_train.npy", y_train.values)
np.save("../data/processed/y_test.npy", y_test.values)

**Save the Scaler**

In [9]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

**Feature Engineering Completed**

Next: 03_model_training.ipynb